In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('../data/train.csv')
meal = pd.read_csv('../data/meal_info.csv')
center = pd.read_csv('../data/fulfilment_center_info.csv')

In [3]:
df = train.merge(meal, on='meal_id')
df = df.merge(center, on='center_id')

In [4]:
df['week'] = pd.to_datetime(
    df['week'],
    unit='D',
    origin='2020-01-01'
)

In [5]:
df.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,2020-01-02,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,2020-01-02,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,2020-01-02,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,2020-01-02,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,2020-01-02,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


In [6]:
df['year'] = df['week'].dt.year

df['month'] = df['week'].dt.month

df['day'] = df['week'].dt.day

df['day_of_week'] = df['week'].dt.dayofweek

df['weekend'] = df['day_of_week'].apply(
    lambda x: 1 if x >= 5 else 0
)

In [7]:
df[['week','year','month','day','day_of_week','weekend']].head()

,week,year,month,day,day_of_week,weekend
0,2020-01-02,2020,1,2,3,0
1,2020-01-02,2020,1,2,3,0
2,2020-01-02,2020,1,2,3,0
3,2020-01-02,2020,1,2,3,0
4,2020-01-02,2020,1,2,3,0


In [8]:
df = df.sort_values(by=['meal_id','week'])

In [9]:
df['lag_1'] = df.groupby('meal_id')['num_orders'].shift(1)

df['lag_7'] = df.groupby('meal_id')['num_orders'].shift(7)

In [10]:
df['rolling_mean_7'] = (
    df.groupby('meal_id')['num_orders']
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

In [11]:
df[['meal_id',
    'week',
    'num_orders',
    'lag_1',
    'lag_7',
    'rolling_mean_7']].head(15)

,meal_id,week,num_orders,lag_1,lag_7,rolling_mean_7
7,1062,2020-01-02,391,NaN,NaN,NaN
40,1062,2020-01-02,514,391.0,NaN,NaN
84,1062,2020-01-02,798,514.0,NaN,NaN
125,1062,2020-01-02,284,798.0,NaN,NaN
164,1062,2020-01-02,256,284.0,NaN,NaN
201,1062,2020-01-02,998,256.0,NaN,NaN
246,1062,2020-01-02,393,998.0,NaN,NaN
287,1062,2020-01-02,485,393.0,391.0,519.142857
332,1062,2020-01-02,393,485.0,514.0,532.571429
365,1062,2020-01-02,163,393.0,798.0,515.285714


In [12]:
df.fillna(0, inplace=True)

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,...,center_type,op_area,year,month,day,day_of_week,weekend,lag_1,lag_7,rolling_mean_7
7,1499955,2020-01-02,55,1062,182.36,183.36,0,0,391,Beverages,...,TYPE_C,2.0,2020,1,2,3,0,0.0,0.0,0.000000
40,1351400,2020-01-02,24,1062,184.36,182.36,0,0,514,Beverages,...,TYPE_B,3.6,2020,1,2,3,0,391.0,0.0,0.000000
84,1057835,2020-01-02,11,1062,184.36,182.36,0,0,798,Beverages,...,TYPE_A,3.7,2020,1,2,3,0,514.0,0.0,0.000000
125,1231510,2020-01-02,83,1062,170.78,170.78,0,0,284,Beverages,...,TYPE_A,5.3,2020,1,2,3,0,798.0,0.0,0.000000
164,1402151,2020-01-02,32,1062,175.60,174.60,0,0,256,Beverages,...,TYPE_A,3.8,2020,1,2,3,0,284.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456255,1061424,2020-05-25,110,2956,582.03,582.03,0,0,41,Fish,...,TYPE_A,3.8,2020,5,25,0,0,95.0,15.0,69.428571
456296,1048620,2020-05-25,132,2956,641.23,640.23,0,0,67,Fish,...,TYPE_A,3.9,2020,5,25,0,0,41.0,53.0,73.142857
456345,1068363,2020-05-25,23,2956,640.23,640.23,0,0,27,Fish,...,TYPE_A,3.4,2020,5,25,0,0,67.0,82.0,75.142857
456429,1378287,2020-05-25,68,2956,581.03,581.03,0,0,121,Fish,...,TYPE_B,4.1,2020,5,25,0,0,27.0,161.0,67.285714


In [13]:
df.isnull().sum()

id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
category                 0
cuisine                  0
city_code                0
region_code              0
center_type              0
op_area                  0
year                     0
month                    0
day                      0
day_of_week              0
weekend                  0
lag_1                    0
lag_7                    0
rolling_mean_7           0
dtype: int64

In [14]:
df = pd.get_dummies(
    df,
    columns=[
        'category',
        'cuisine',
        'center_type'
    ],
    drop_first=True
)

In [15]:
df.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,city_code,...,category_Salad,category_Sandwich,category_Seafood,category_Soup,category_Starters,cuisine_Indian,cuisine_Italian,cuisine_Thai,center_type_TYPE_B,center_type_TYPE_C
7,1499955,2020-01-02,55,1062,182.36,183.36,0,0,391,647,...,False,False,False,False,False,False,True,False,False,True
40,1351400,2020-01-02,24,1062,184.36,182.36,0,0,514,614,...,False,False,False,False,False,False,True,False,True,False
84,1057835,2020-01-02,11,1062,184.36,182.36,0,0,798,679,...,False,False,False,False,False,False,True,False,False,False
125,1231510,2020-01-02,83,1062,170.78,170.78,0,0,284,659,...,False,False,False,False,False,False,True,False,False,False
164,1402151,2020-01-02,32,1062,175.60,174.60,0,0,256,526,...,False,False,False,False,False,False,True,False,False,False


In [16]:
y = df['num_orders']

In [17]:
X = df.drop(['num_orders'], axis=1)

In [18]:
X = X.drop(['week'], axis=1)

In [19]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]

X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]

y_test = y.iloc[split_index:]


In [20]:
print(X_train.shape)

print(X_test.shape)

print(y_train.shape)

print(y_test.shape)

(365238, 36)
(91310, 36)
(365238,)
(91310,)


In [21]:
df.to_csv('../data/processed_data.csv', index=False)